# Model Version Comparison

Compare performance between different churn prediction model versions.

This notebook:
- Loads all model versions from Snowflake
- Compares metrics (F1, Precision, Recall, AUC)
- Visualizes performance differences
- Identifies the best performing model

## 1. Setup and Imports

In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from snowflake.snowpark.context import get_active_session

print("="*70)
print("MODEL VERSION COMPARISON")
print("="*70)

# ============================================================================
# CONFIGURATION: Choose where to load models from
# ============================================================================
# Set to 'snowflake' to load from Snowflake MODEL_REGISTRY
# Set to 'local' to load from local folder
LOAD_MODE = 'local'  # Change this to 'snowflake' if models are registered

# If using local mode, specify the folder containing model files
LOCAL_MODEL_FOLDER = '.'  # Current directory (change if models are elsewhere)

print(f"\n📍 Load Mode: {LOAD_MODE.upper()}")

if LOAD_MODE == 'snowflake':
    # Get Snowflake session
    session = get_active_session()
    session.use_database("MY_DATABASE")
    session.use_schema("PUBLIC")
    print(f"✓ Connected to Snowflake")
    print(f"  Database: {session.get_current_database()}")
    print(f"  Schema: {session.get_current_schema()}")
else:
    print(f"✓ Using local folder: {LOCAL_MODEL_FOLDER}")
    session = None

## 2. Load Model Versions (Snowflake or Local)

In [ ]:
models = {}

if LOAD_MODE == 'snowflake':
    # ========================================================================
    # LOAD FROM SNOWFLAKE MODEL_REGISTRY
    # ========================================================================
    print("\n📥 Loading models from Snowflake MODEL_REGISTRY...")
    
    try:
        versions_df = session.sql("""
            SELECT VERSION, MODEL_PATH, DESCRIPTION
            FROM MY_DATABASE.PUBLIC.MODEL_REGISTRY 
            ORDER BY VERSION
        """).to_pandas()
        
        print(f"Found {len(versions_df)} version(s) in registry:")
        for _, row in versions_df.iterrows():
            print(f"  - {row['VERSION']}: {row['DESCRIPTION'][:60]}...")
        
        # Download and load each version
        for _, row in versions_df.iterrows():
            version = row['VERSION']
            print(f"\n  Loading {version}...")
            
            try:
                # Download from Snowflake
                session.file.get(
                    f'@MY_DATABASE.PUBLIC.MODELS/{version}/churn_model_{version}.pkl',
                    '/tmp/'
                )
                
                # Load pickle
                with open(f'/tmp/churn_model_{version}.pkl', 'rb') as f:
                    models[version] = pickle.load(f)
                
                print(f"    ✓ {version} loaded successfully")
            except Exception as e:
                print(f"    ✗ Error loading {version}: {e}")
        
        print(f"\n✓ Successfully loaded {len(models)} model(s) from Snowflake")
        
    except Exception as e:
        print(f"⚠️ Error reading MODEL_REGISTRY: {e}")
        print("\nTip: Make sure you've run the training notebooks and registered the models.")

else:
    # ========================================================================
    # LOAD FROM LOCAL FOLDER
    # ========================================================================
    print("\n📂 Loading models from local folder...")
    
    # Find all .pkl files matching pattern churn_model_*.pkl
    pattern = os.path.join(LOCAL_MODEL_FOLDER, 'churn_model_v*.pkl')
    model_files = glob.glob(pattern)
    
    if not model_files:
        print(f"⚠️ No model files found matching pattern: {pattern}")
        print(f"\nLooking for files like:")
        print(f"  - churn_model_v1.pkl")
        print(f"  - churn_model_v2.pkl")
        print(f"  - churn_model_v3.pkl")
        print(f"\nMake sure model files are in: {os.path.abspath(LOCAL_MODEL_FOLDER)}")
    else:
        print(f"Found {len(model_files)} model file(s):")
        for file_path in sorted(model_files):
            print(f"  - {os.path.basename(file_path)}")
        
        # Load each model
        for file_path in sorted(model_files):
            # Extract version from filename (e.g., "churn_model_v1.pkl" -> "v1")
            filename = os.path.basename(file_path)
            version = filename.replace('churn_model_', '').replace('.pkl', '')
            
            print(f"\n  Loading {version}...")
            
            try:
                with open(file_path, 'rb') as f:
                    models[version] = pickle.load(f)
                
                # Display metadata
                if 'metadata' in models[version]:
                    desc = models[version]['metadata'].get('description', 'No description')
                    print(f"    ✓ {version}: {desc[:60]}")
                else:
                    print(f"    ✓ {version} loaded")
                    
            except Exception as e:
                print(f"    ✗ Error loading {version}: {e}")
        
        print(f"\n✓ Successfully loaded {len(models)} model(s) from local folder")

# Summary
if not models:
    print("\n❌ No models loaded. Cannot proceed with comparison.")
    print("\nTroubleshooting:")
    if LOAD_MODE == 'local':
        print(f"  1. Check that model files exist in: {os.path.abspath(LOCAL_MODEL_FOLDER)}")
        print(f"  2. Files should be named: churn_model_v1.pkl, churn_model_v2.pkl, etc.")
        print(f"  3. Train models first by running the training notebooks")
    else:
        print(f"  1. Check that MODEL_REGISTRY table exists in Snowflake")
        print(f"  2. Train models first by running the training notebooks")
        print(f"  3. Make sure models were uploaded to Snowflake stage")
else:
    print(f"\n📊 Ready to compare {len(models)} version(s): {', '.join(sorted(models.keys()))}")

In [ ]:
# This cell is no longer needed - loading logic moved to previous cell
# Kept for backwards compatibility
pass

## 3. Compare Metrics

In [ ]:
if models:
    print("\n" + "="*70)
    print("PERFORMANCE COMPARISON")
    print("="*70)
    
    # Build comparison table
    comparison_data = []
    for version, model in models.items():
        comparison_data.append({
            'Version': version,
            'Description': model['metadata']['description'][:40] + '...',
            'F1': f"{model['test_metrics']['f1']:.4f}",
            'Precision': f"{model['test_metrics']['precision']:.4f}",
            'Recall': f"{model['test_metrics']['recall']:.4f}",
            'AUC': f"{model['test_metrics']['auc']:.4f}",
            'Loss': f"{model['test_metrics']['loss']:.4f}",
            'Train Date': model['metadata']['train_date'][:10]
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    print("\n" + comparison_df.to_string(index=False))
    
    # Find best model (by F1)
    f1_scores = {v: m['test_metrics']['f1'] for v, m in models.items()}
    best_version = max(f1_scores, key=f1_scores.get)
    best_f1 = f1_scores[best_version]
    
    print("\n" + "="*70)
    print(f"🏆 BEST MODEL: {best_version}")
    print(f"   F1 Score: {best_f1:.4f}")
    print("="*70)
else:
    print("\n⚠️ No models to compare")

## 4. Visualize Performance Comparison

In [ ]:
if models:
    # Prepare data for visualization
    versions = list(models.keys())
    metrics_data = {
        'F1': [models[v]['test_metrics']['f1'] for v in versions],
        'Precision': [models[v]['test_metrics']['precision'] for v in versions],
        'Recall': [models[v]['test_metrics']['recall'] for v in versions],
        'AUC': [models[v]['test_metrics']['auc'] for v in versions]
    }
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle('Model Version Comparison', fontsize=16, fontweight='bold')
    
    metrics = ['F1', 'Precision', 'Recall', 'AUC']
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
    
    for idx, (metric, color) in enumerate(zip(metrics, colors)):
        ax = axes[idx // 2, idx % 2]
        
        bars = ax.bar(versions, metrics_data[metric], color=color, alpha=0.7, edgecolor='black')
        
        # Highlight best
        best_idx = metrics_data[metric].index(max(metrics_data[metric]))
        bars[best_idx].set_alpha(1.0)
        bars[best_idx].set_linewidth(3)
        
        ax.set_ylabel(metric, fontweight='bold', fontsize=11)
        ax.set_xlabel('Version', fontweight='bold', fontsize=11)
        ax.set_title(f'{metric} Score', fontweight='bold', fontsize=12)
        ax.set_ylim([0, 1])
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        
        # Add value labels on bars
        for bar, value in zip(bars, metrics_data[metric]):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{value:.4f}',
                   ha='center', va='bottom', fontweight='bold', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Comparison visualized!")
else:
    print("⚠️ No models to visualize")

## 5. Detailed Version Information

In [ ]:
for version, model in models.items():
    print("\n" + "="*70)
    print(f"VERSION: {version}")
    print("="*70)
    print(f"Description: {model['metadata']['description']}")
    print(f"Train Date: {model['metadata']['train_date']}")
    
    print(f"\n📊 Performance Metrics:")
    print(f"  F1 Score:  {model['test_metrics']['f1']:.4f}")
    print(f"  Precision: {model['test_metrics']['precision']:.4f}")
    print(f"  Recall:    {model['test_metrics']['recall']:.4f}")
    print(f"  AUC-ROC:   {model['test_metrics']['auc']:.4f}")
    print(f"  Loss:      {model['test_metrics']['loss']:.4f}")
    
    print(f"\n🔧 Model Configuration:")
    print(f"  Features: {model['metadata']['num_features']}")
    print(f"  Lookback window: {model['metadata']['lookback_window']} months")
    print(f"  Hidden size: {model['model_config']['hidden_size']}")
    print(f"  Num layers: {model['model_config']['num_layers']}")
    print(f"  Dropout: {model['model_config']['dropout']}")
    print(f"  Threshold: {model['threshold']:.2f}")
    
    print(f"\n📈 Training Info:")
    print(f"  Batch size: {model['metadata']['batch_size']}")
    print(f"  Learning rate: {model['metadata']['learning_rate']}")
    print(f"  Epochs trained: {model['metadata']['num_epochs_trained']}")
    print(f"  Train samples: {model['metadata']['train_samples']:,}")
    print(f"  Test samples: {model['metadata']['test_samples']:,}")
    
    # Show changes from v1 if available
    if 'changes_from_v1' in model['metadata']:
        print(f"\n📝 Changes from v1:")
        changes = model['metadata']['changes_from_v1']
        if isinstance(changes, list):
            for change in changes:
                print(f"  • {change}")
        else:
            print(f"  • {changes}")

## 6. Performance Improvement Analysis

In [ ]:
if len(models) >= 2 and 'v1' in models and 'v2' in models:
    print("\n" + "="*70)
    print("V2 vs V1 IMPROVEMENT ANALYSIS")
    print("="*70)
    
    v1_metrics = models['v1']['test_metrics']
    v2_metrics = models['v2']['test_metrics']
    
    improvements = {}
    for metric in ['f1', 'precision', 'recall', 'auc']:
        v1_val = v1_metrics[metric]
        v2_val = v2_metrics[metric]
        improvement = ((v2_val - v1_val) / v1_val) * 100
        improvements[metric] = improvement
        
        symbol = "📈" if improvement > 0 else "📉" if improvement < 0 else "➡️"
        print(f"\n{metric.upper()}:")
        print(f"  v1: {v1_val:.4f}")
        print(f"  v2: {v2_val:.4f}")
        print(f"  {symbol} Change: {improvement:+.2f}%")
    
    # Overall verdict
    avg_improvement = sum(improvements.values()) / len(improvements)
    print("\n" + "="*70)
    if avg_improvement > 5:
        print("✅ VERDICT: v2 shows significant improvement!")
        print(f"   Average improvement: {avg_improvement:.2f}%")
    elif avg_improvement > 0:
        print("✅ VERDICT: v2 shows modest improvement")
        print(f"   Average improvement: {avg_improvement:.2f}%")
    elif avg_improvement > -5:
        print("⚠️ VERDICT: Performance similar between versions")
        print(f"   Average change: {avg_improvement:.2f}%")
    else:
        print("❌ VERDICT: v1 performs better")
        print(f"   Average decline: {avg_improvement:.2f}%")
    print("="*70)
elif len(models) == 1:
    print("\nℹ️ Only one version available. Train additional versions to compare.")
else:
    print("\nℹ️ Need v1 and v2 for improvement analysis.")

## 7. Recommendation

In [ ]:
if models:
    print("\n" + "="*70)
    print("RECOMMENDATION")
    print("="*70)
    
    # Find best by F1
    f1_scores = {v: m['test_metrics']['f1'] for v, m in models.items()}
    best_version = max(f1_scores, key=f1_scores.get)
    
    print(f"\n🏆 Use {best_version} for production")
    print(f"\n📊 {best_version.upper()} Metrics:")
    print(f"  F1 Score:  {models[best_version]['test_metrics']['f1']:.4f}")
    print(f"  Precision: {models[best_version]['test_metrics']['precision']:.4f}")
    print(f"  Recall:    {models[best_version]['test_metrics']['recall']:.4f}")
    print(f"  AUC:       {models[best_version]['test_metrics']['auc']:.4f}")
    
    print(f"\n💡 To use this model:")
    print(f"   1. Load from: @MY_DATABASE.PUBLIC.MODELS/{best_version}/churn_model_{best_version}.pkl")
    print(f"   2. Set IS_PRODUCTION = TRUE in MODEL_REGISTRY for {best_version}")
    print(f"   3. Update prediction pipeline to use {best_version}")
    
    print("\n" + "="*70)